# 39 · Total within groups, with each group's share, against pandas

**Question.** Fieldwork experiment 191 gave Total within groups a share: each group's total or count, and beside it its
percentage of the total of the groups agreeing on the columns named *within*, or of all of them. On check 29's South
London table, does pandas get every total and every share, and the same unknowns where a cell is empty?

The totals are integers and the rounding is declared, so the tolerance is **zero**.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")

## Parity

Three cases: each company's share of the population and of the deaths; rows counted by company, each a share of all;
and one population cell emptied, so that every population share is unknown and the deaths' shares are unchanged.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_39_group_shares import run

report = run(Path.cwd() / "fixtures" / "group-shares-snow-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    print(f"\n{'OK ' if c['agrees'] else 'DIFF'} case {c['case']}: {c['note']}")
    for cell in c["cells"]:
        print(f"  {'OK ' if cell['agrees'] else 'DIFF'} {' · '.join(cell['key'])}: {cell['pandas']} / {cell['fieldwork']}")
    for stratum, s in c["strataTotals"].items():
        print(f"  shares in {stratum} total {s['total']} over {s['groups']} groups")
print("\nempty cell:", report["emptyCell"])

## Rounding is a rule, not a detail

The share is held to one decimal place, half up, so the population shares of case 1 total **99.9**, not 100. Python's
own `round(x, 1)` rounds half to even on the binary value; the check records where it would land elsewhere, because a
lab that reached for it would report a difference of rule as a defect.

In [ ]:
for c in report["comparisons"]:
    print(f"case {c['case']}: round() would differ on {len(c['pythonRoundWouldDiffer'])} share(s)")

## What this cannot establish

**That a share is the measure a reader needs.** A share of rows is not a rate, and a share of deaths by company says
nothing of risk without the population beside it, which is why case 1 shares both. **That one decimal place is
enough.**

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas  :", report["environment"]["pandas"])
print("runtime :", report["environment"]["runtime"])